# Intro to MLOps — hands-on (notebook)

Notebook version of `train.py`. Track an ML experiment with MLflow, then
compare runs in the UI.

- Tutorial: <https://jienweng.github.io/notes/intro-to-mlops-tutorial/>
- Slides:   <https://jienweng.github.io/slides/2026/intro-to-mlops/>

**How to use this notebook**
1. `pip install -r requirements.txt`
2. Run all cells. One MLflow run is logged.
3. Change `C` (e.g. to `0.01`), run again — now you have a second run.
4. In a terminal: `mlflow ui --backend-store-uri sqlite:///mlflow.db`,
   open <http://127.0.0.1:5000>, and **compare the two runs**.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score
from sklearn.model_selection import train_test_split

# Local SQLite store (recommended on MLflow 3.x); artifacts go to ./mlruns
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('intro-to-mlops')

## Data

In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

## Train, evaluate, and log one run

Change `C` and re-run this cell to produce a second run to compare.

In [ ]:
C = 1.0  # inverse regularisation strength — try 0.01 on a second run

with mlflow.start_run(run_name=f'logreg-C={C}'):
    model = LogisticRegression(C=C, max_iter=10_000)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)

    mlflow.log_param('model', 'LogisticRegression')
    mlflow.log_param('C', C)
    mlflow.log_metric('accuracy', acc)
    mlflow.log_metric('f1', f1)
    mlflow.sklearn.log_model(model, name='model')

    ConfusionMatrixDisplay.from_predictions(y_test, preds)
    plt.title(f'Confusion matrix (C={C})')
    plt.savefig('confusion_matrix.png', bbox_inches='tight')
    mlflow.log_artifact('confusion_matrix.png')

    print(f'Logged run: accuracy={acc:.4f}, f1={f1:.4f}')

## Compare runs

After logging at least two runs, launch the UI from a terminal:

```shell
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

Open <http://127.0.0.1:5000>, select the two runs, and click **Compare**.

**Tip:** add `mlflow.sklearn.autolog()` before `fit()` to capture
params, metrics, and the model automatically in one line.